# Sanity Check - Step 07: Epoching

Überprüft:
- Epochs erfolgreich erstellt
- Event-Anzahl und -Typen
- Epoch-Größe (Anzahl und Dimensionen)
- Baseline-Korrektur

In [ ]:
import sys
from pathlib import Path
import mne
import numpy as np

sys.path.append(str(Path.cwd().parent / 'eeg_pipeline'))
import config

print("Setup erfolgreich")

## 1. Epochs laden

In [ ]:
subject_id = config.SUBJECTS[0]
person = "P1"

epoch_path = config.OUTPUT_DIR / f"sub-{subject_id}_{person}_epoch.fif"

if epoch_path.exists():
    epochs = mne.read_epochs(str(epoch_path), preload=False)
    print(f"✓ Epochs loaded for {person}")
else:
    print(f"✗ Epoch file not found")

## 2. Epoch-Grundlagen

In [ ]:
print(f"=== EPOCH DETAILS ===")
print(f"Number of epochs: {len(epochs)}")
print(f"Number of channels: {len(epochs.ch_names)}")
print(f"Samples per epoch: {epochs.get_data().shape[2]}")
print(f"Sampling rate: {epochs.info['sfreq']} Hz")

print(f"\nEvent types: {epochs.event_id}")
print(f"Time window: [{epochs.times[0]:.3f}, {epochs.times[-1]:.3f}] s")
print(f"Expected duration: {config.EPOCH_TMAX - config.EPOCH_TMIN:.3f}s")
print(f"Actual duration: {epochs.times[-1] - epochs.times[0]:.3f}s")

## 3. Baseline & Data Quality

In [ ]:
print(f"=== DATA QUALITY ===")
if epochs.baseline is not None:
    print(f"✓ Baseline applied: {epochs.baseline}")
else:
    print(f"⚠ No baseline correction")

data = epochs.get_data()
nan_count = int(np.isnan(data).sum())
inf_count = int(np.isinf(data).sum())

if nan_count == 0 and inf_count == 0:
    print(f"✓ No NaN or Inf values")
else:
    print(f"✗ Found {nan_count} NaN and {inf_count} Inf values")

bads = epochs.info.get('bads', [])
if len(bads) == 0:
    print(f"✓ No bad channels marked")
else:
    print(f"⚠ Bad channels marked: {len(bads)}")

## 4. Event-Verteilung

In [ ]:
if len(epochs.event_id) > 0:
    print(f"Event Distribution:")
    event_counts = epochs.event_id
    # Create a summary
    for event_name in epochs.event_id.keys():
        try:
            count = len(epochs[event_name])
            print(f"  {event_name}: {count} epochs")
        except:
            print(f"  {event_name}: (could not count)")
else:
    print(f"No events found in epochs")